In [1]:
# 1. Import libraries
import pandas as pd
import re
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# 2. Load dataset
df = pd.read_csv("wound_visits.csv")

print(df.head())
print(df.columns)

# 3. Select text column
# Change "notes" to your actual text column name
text_column = "notes"

df[text_column] = df[text_column].fillna("")

# 4. NLP text cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    return " ".join(text.split())

df["clean_text"] = df[text_column].apply(clean_text)

# 5. Convert text to TF-IDF
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=2000
)

X = vectorizer.fit_transform(df["clean_text"])

# 6. Apply K-Means
k = 4

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X)

# 7. Display cluster sizes
print("\nCluster sizes:")
print(df["cluster"].value_counts().sort_index())

# 8. Display important words in each cluster
terms = vectorizer.get_feature_names_out()

for i in range(k):
    words = kmeans.cluster_centers_[i].argsort()[-10:][::-1]
    print(f"\nCluster {i}:")
    print(", ".join(terms[words]))

# 9. Visualize clusters
X_pca = PCA(n_components=2).fit_transform(X.toarray())

plt.figure(figsize=(8, 5))
plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=df["cluster"],
    cmap="viridis"
)

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("K-Means Clustering of Wound Visits")
plt.colorbar(label="Cluster")
plt.show()

# 10. Save results
df.to_csv("wound_visits_clustered.csv", index=False)

print("\nClustering completed!")

  patient_id  visit_date           wound_type  wound_size_cm2  \
0       P001  2025-01-05  Diabetic Foot Ulcer             6.2   
1       P001  2025-01-19  Diabetic Foot Ulcer             5.4   
2       P001  2025-02-02  Diabetic Foot Ulcer             4.8   
3       P002  2025-01-08     Venous Leg Ulcer             9.5   
4       P002  2025-01-22     Venous Leg Ulcer             9.9   

             treatment                                     clinical_notes  
0    Sharp debridement  Ulcer on plantar surface with moderate serous ...  
1    Advanced dressing  Wound size reduced with healthy granulation ti...  
2    Advanced dressing  Continued improvement. Minimal drainage. Surro...  
3  Compression therapy  Shallow ulcer with heavy exudate and surroundi...  
4  Compression therapy  Slight increase in wound size. Macerated edges...  
Index(['patient_id', 'visit_date', 'wound_type', 'wound_size_cm2', 'treatment',
       'clinical_notes'],
      dtype='str')


KeyError: 'notes'